# BERT Embedding Pipeline (Smoke Test)

This notebook verifies that the BERT embedding pipeline works correctly before launching the full-scale job with a bash script.

## Goals
1. Verify data loading
2. Load a pretrained BERT model
3. Extract word-level embeddings for a small subset of stories
4. Apply preprocessing
5. Check output shapes
6. Save outputs in the same format as the full pipeline

If everything works here, the pipeline is ready for full-scale execution on Bridges2.

In [1]:
import os
import sys
import json
import pickle
import random
from pathlib import Path
from typing import Dict, List

import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel

/jet/home/chsu13/.conda/envs/stat214/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Make sure the notebook can import preprocessing_utils.py
if 'code' not in sys.path:
    sys.path.append('code')

sys.path.append(os.path.abspath('..'))

from preprocessing_utils import preprocess_embeddings

In [3]:
# =========================
# Config (Smoke Test Mode)
# =========================

# Bridges2 shared data path
DATA_PATH = '/ocean/projects/mth250011p/shared/215a/final_project/data'

# model
MODEL_NAME = 'google-bert/bert-base-uncased'
CACHE_DIR = os.environ.get('HF_HOME', os.path.expanduser('~/.cache/huggingface'))

# output
OUTPUT_DIR = '../../data/embeddings_smoke_test'

# subjects
SUBJECTS = ['subject2', 'subject3']
SUBJECT_IDS = {'subject2': 's2', 'subject3': 's3'}

# smoke test settings
N_TRAIN_STORIES = 1
N_TEST_STORIES = 1

MAX_WORDS_PER_CHUNK = 128
BATCH_SIZE = 2
LAYER = 'last_hidden_state'   # choices: 'last_hidden_state', 'last4_mean'
POOLING = 'mean'              # choices: 'mean', 'first'
SAVE_STORY_LEVEL = True
SEED = 42

# device
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Device:", DEVICE)
print("Data path:", DATA_PATH)
print("Output dir:", OUTPUT_DIR)
print("Cache dir:", CACHE_DIR)

Device: cuda
Data path: /ocean/projects/mth250011p/shared/215a/final_project/data
Output dir: ../../data/embeddings_smoke_test
Cache dir: /jet/home/chsu13/.cache/huggingface


In [4]:
print("DATA_PATH exists:", os.path.exists(DATA_PATH))
print("raw_text.pkl exists:", os.path.exists(f'{DATA_PATH}/raw_text.pkl'))
print("local data/train_test_split.json exists:", os.path.exists('../../data/train_test_split.json'))

DATA_PATH exists: True
raw_text.pkl exists: True
local data/train_test_split.json exists: True


In [5]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def chunk_list(items: List[str], chunk_size: int) -> List[List[str]]:
    return [items[i:i + chunk_size] for i in range(0, len(items), chunk_size)]

seed_everything(SEED)

In [6]:
print("Loading raw text...")
with open(f'{DATA_PATH}/raw_text.pkl', 'rb') as f:
    raw_text = pickle.load(f)

print("Loading train/test split...")

# Prefer split file under DATA_PATH if available; otherwise use local project file
split_json_path = f'{DATA_PATH}/train_test_split.json'
if not os.path.exists(split_json_path):
    split_json_path = '../../data/train_test_split.json'

with open(split_json_path, 'r') as f:
    split = json.load(f)

train_stories_all = split['train']
test_stories_all = split['test']

train_stories = train_stories_all[:N_TRAIN_STORIES]
test_stories = test_stories_all[:N_TEST_STORIES]
all_stories = train_stories + test_stories

print("Split file:", split_json_path)
print("Total available train stories:", len(train_stories_all))
print("Total available test stories:", len(test_stories_all))
print("Smoke test train stories:", train_stories)
print("Smoke test test stories:", test_stories)

Loading raw text...
Loading train/test split...
Split file: ../../data/train_test_split.json
Total available train stories: 81
Total available test stories: 20
Smoke test train stories: ['mybackseatviewofagreatromance']
Smoke test test stories: ['itsabox']


/var/tmp/ipykernel_28980/2936895409.py:3: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  raw_text = pickle.load(f)


In [7]:
def load_bert(model_name: str, cache_dir: str, device: str):
    print(f'Loading tokenizer from {model_name}')
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        use_fast=True,
        cache_dir=cache_dir,
    )

    print(f'Loading model from {model_name}')
    model = AutoModel.from_pretrained(
        model_name,
        cache_dir=cache_dir,
    )

    model.eval()
    model.to(device)

    hidden_size = model.config.hidden_size
    print("Hidden size:", hidden_size)
    print("Using device:", device)
    return tokenizer, model, hidden_size

tokenizer, model, hidden_size = load_bert(
    model_name=MODEL_NAME,
    cache_dir=CACHE_DIR,
    device=DEVICE,
)

Loading tokenizer from google-bert/bert-base-uncased
Loading model from google-bert/bert-base-uncased


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7318.23it/s]
BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Hidden size: 768
Using device: cuda


In [8]:
def words_to_bert_embeddings(
    words: List[str],
    tokenizer,
    model,
    device: str,
    max_words_per_chunk: int = 128,
    batch_size: int = 2,
    layer: str = 'last_hidden_state',
    pooling: str = 'mean',
) -> np.ndarray:
    """
    Return one contextual embedding per original word.

    Steps:
    1. Split words into manageable chunks
    2. Tokenize with is_split_into_words=True
    3. Run BERT
    4. Map subword token representations back to original words
    5. Pool subwords into a single vector per word
    """
    if len(words) == 0:
        raise ValueError("Received empty word list.")

    word_chunks = chunk_list(words, max_words_per_chunk)
    story_word_vectors = []

    with torch.no_grad():
        for batch_start in range(0, len(word_chunks), batch_size):
            batch_chunks = word_chunks[batch_start: batch_start + batch_size]

            encoding = tokenizer(
                batch_chunks,
                is_split_into_words=True,
                return_tensors='pt',
                padding=True,
                truncation=True,
                max_length=512,
                return_attention_mask=True,
                return_token_type_ids=True,
            )

            # Save fast-tokenizer metadata before moving tensors to device
            batch_encodings = encoding.encodings
            encoding = {k: v.to(device) for k, v in encoding.items()}

            if layer == 'last4_mean':
                outputs = model(**encoding, output_hidden_states=True)
                stacked = torch.stack(outputs.hidden_states[-4:], dim=0)  # (4, B, T, H)
                token_reps = stacked.mean(dim=0)  # (B, T, H)
            else:
                outputs = model(**encoding)
                token_reps = outputs.last_hidden_state  # (B, T, H)

            token_reps = token_reps.detach().cpu()

            for i, words_in_chunk in enumerate(batch_chunks):
                word_ids = batch_encodings[i].word_ids
                reps_i = token_reps[i]  # (T, H)
                n_words = len(words_in_chunk)

                per_word = [[] for _ in range(n_words)]
                for token_idx, word_id in enumerate(word_ids):
                    if word_id is None:
                        continue
                    per_word[word_id].append(reps_i[token_idx])

                out = []
                for piece_list in per_word:
                    if len(piece_list) == 0:
                        out.append(torch.zeros(reps_i.shape[-1], dtype=reps_i.dtype))
                    elif pooling == 'first':
                        out.append(piece_list[0])
                    else:
                        out.append(torch.stack(piece_list, dim=0).mean(dim=0))

                out = torch.stack(out, dim=0).numpy().astype(np.float32)
                story_word_vectors.append(out)

    X_story = np.vstack(story_word_vectors)

    if X_story.shape[0] != len(words):
        raise RuntimeError(
            f'Word count mismatch after BERT extraction: got {X_story.shape[0]} rows '
            f'but expected {len(words)}.'
        )

    return X_story

In [9]:
def build_story_level_bert_vectors(
    raw_text: Dict,
    stories: List[str],
    tokenizer,
    model,
    device: str,
    max_words_per_chunk: int,
    batch_size: int,
    layer: str,
    pooling: str,
    save_story_level_dir: str = None,
) -> Dict[str, np.ndarray]:
    """
    Generate word-level BERT embeddings for each story.

    Returns:
        dict: {story_name: np.ndarray of shape (num_words, hidden_size)}
    """
    bert_vectors = {}
    total = len(stories)

    for idx, story in enumerate(stories, start=1):
        words = raw_text[story].data
        print(f'[{idx}/{total}] Extracting embeddings for story="{story}" (n_words={len(words)})')

        X_story = words_to_bert_embeddings(
            words=words,
            tokenizer=tokenizer,
            model=model,
            device=device,
            max_words_per_chunk=max_words_per_chunk,
            batch_size=batch_size,
            layer=layer,
            pooling=pooling,
        )

        bert_vectors[story] = X_story
        print("    word-level shape:", X_story.shape)

        if save_story_level_dir is not None:
            os.makedirs(save_story_level_dir, exist_ok=True)
            out_path = os.path.join(save_story_level_dir, f'{story}_pretrained_bert_wordlevel.npz')
            np.savez_compressed(out_path, X=X_story)
            print("    saved to:", out_path)

    return bert_vectors

In [10]:
story_level_dir = None
if SAVE_STORY_LEVEL:
    story_level_dir = os.path.join(OUTPUT_DIR, 'story_level_pretrained_bert')

bert_vectors = build_story_level_bert_vectors(
    raw_text=raw_text,
    stories=all_stories,
    tokenizer=tokenizer,
    model=model,
    device=DEVICE,
    max_words_per_chunk=MAX_WORDS_PER_CHUNK,
    batch_size=BATCH_SIZE,
    layer=LAYER,
    pooling=POOLING,
    save_story_level_dir=story_level_dir,
)

[1/2] Extracting embeddings for story="mybackseatviewofagreatromance" (n_words=1794)
    word-level shape: (1794, 768)
    saved to: ../../data/embeddings_smoke_test/story_level_pretrained_bert/mybackseatviewofagreatromance_pretrained_bert_wordlevel.npz
[2/2] Extracting embeddings for story="itsabox" (n_words=1708)
    word-level shape: (1708, 768)
    saved to: ../../data/embeddings_smoke_test/story_level_pretrained_bert/itsabox_pretrained_bert_wordlevel.npz


In [11]:
example_story = all_stories[0]

print("Example story:", example_story)
print("Original number of words:", len(raw_text[example_story].data))
print("Word-level embedding shape:", bert_vectors[example_story].shape)
print("First embedding vector (first 8 dims):")
print(bert_vectors[example_story][0][:8])

Example story: mybackseatviewofagreatromance
Original number of words: 1794
Word-level embedding shape: (1794, 768)
First embedding vector (first 8 dims):
[0. 0. 0. 0. 0. 0. 0. 0.]


In [12]:
for split_name, stories in [('train', train_stories), ('test', test_stories)]:
    print(f'\n=== Preprocessing {split_name} split ===')

    processed = preprocess_embeddings(
        stories=stories,
        word_vectors=bert_vectors,
        wordseqs=raw_text,
    )

    for story in stories:
        print(f'{story}: processed shape = {processed[story].shape}')

    X = np.vstack([processed[story] for story in stories]).astype(np.float32)
    print(f'Final stacked X shape for {split_name}: {X.shape}')


=== Preprocessing train split ===
mybackseatviewofagreatromance: processed shape = (384, 3072)
Final stacked X shape for train: (384, 3072)

=== Preprocessing test split ===
itsabox: processed shape = (355, 3072)
Final stacked X shape for test: (355, 3072)


In [13]:
metadata = {
    'model_name': MODEL_NAME,
    'hidden_size': hidden_size,
    'layer': LAYER,
    'pooling': POOLING,
    'max_words_per_chunk': MAX_WORDS_PER_CHUNK,
    'batch_size': BATCH_SIZE,
    'subjects': SUBJECTS,
    'smoke_test': True,
    'n_train_stories': N_TRAIN_STORIES,
    'n_test_stories': N_TEST_STORIES,
    'data_path': DATA_PATH,
}

metadata_path = os.path.join(OUTPUT_DIR, 'pretrained_bert_embedding_metadata.json')
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print("Saved metadata to:", metadata_path)

Saved metadata to: ../../data/embeddings_smoke_test/pretrained_bert_embedding_metadata.json


In [14]:
for subject in SUBJECTS:
    sid = SUBJECT_IDS[subject]
    print(f'\nProcessing {subject} ({sid})...')

    for split_name, stories in [('train', train_stories), ('test', test_stories)]:
        processed = preprocess_embeddings(
            stories=stories,
            word_vectors=bert_vectors,
            wordseqs=raw_text,
        )

        X = np.vstack([processed[story] for story in stories]).astype(np.float32)

        out_path = os.path.join(OUTPUT_DIR, f'{sid}_{split_name}_pretrained_bert_embeddings.npz')
        np.savez_compressed(out_path, X=X)

        split_meta = {
            'subject': subject,
            'subject_id': sid,
            'split': split_name,
            'stories': stories,
            'shape': list(X.shape),
            'feature_dim_before_delay': hidden_size,
            'feature_dim_after_delay': X.shape[1],
            'smoke_test': True,
        }

        json_path = os.path.join(OUTPUT_DIR, f'{sid}_{split_name}_pretrained_bert_embeddings.json')
        with open(json_path, 'w') as f:
            json.dump(split_meta, f, indent=2)

        print(f'Saved {split_name} -> {out_path}, shape = {X.shape}')
        print(f'Saved metadata -> {json_path}')


Processing subject2 (s2)...
Saved train -> ../../data/embeddings_smoke_test/s2_train_pretrained_bert_embeddings.npz, shape = (384, 3072)
Saved metadata -> ../../data/embeddings_smoke_test/s2_train_pretrained_bert_embeddings.json
Saved test -> ../../data/embeddings_smoke_test/s2_test_pretrained_bert_embeddings.npz, shape = (355, 3072)
Saved metadata -> ../../data/embeddings_smoke_test/s2_test_pretrained_bert_embeddings.json

Processing subject3 (s3)...
Saved train -> ../../data/embeddings_smoke_test/s3_train_pretrained_bert_embeddings.npz, shape = (384, 3072)
Saved metadata -> ../../data/embeddings_smoke_test/s3_train_pretrained_bert_embeddings.json
Saved test -> ../../data/embeddings_smoke_test/s3_test_pretrained_bert_embeddings.npz, shape = (355, 3072)
Saved metadata -> ../../data/embeddings_smoke_test/s3_test_pretrained_bert_embeddings.json


In [15]:
print("\nSaved files:")
for root, dirs, files in os.walk(OUTPUT_DIR):
    for file in sorted(files):
        print(os.path.join(root, file))


Saved files:
../../data/embeddings_smoke_test/pretrained_bert_embedding_metadata.json
../../data/embeddings_smoke_test/s2_test_pretrained_bert_embeddings.json
../../data/embeddings_smoke_test/s2_test_pretrained_bert_embeddings.npz
../../data/embeddings_smoke_test/s2_train_pretrained_bert_embeddings.json
../../data/embeddings_smoke_test/s2_train_pretrained_bert_embeddings.npz
../../data/embeddings_smoke_test/s3_test_pretrained_bert_embeddings.json
../../data/embeddings_smoke_test/s3_test_pretrained_bert_embeddings.npz
../../data/embeddings_smoke_test/s3_train_pretrained_bert_embeddings.json
../../data/embeddings_smoke_test/s3_train_pretrained_bert_embeddings.npz
../../data/embeddings_smoke_test/story_level_pretrained_bert/itsabox_pretrained_bert_wordlevel.npz
../../data/embeddings_smoke_test/story_level_pretrained_bert/mybackseatviewofagreatromance_pretrained_bert_wordlevel.npz


In [16]:
# =========================
# CHECK X / Y ALIGNMENT
# =========================

subject = 'subject2'  # change if needed

print("========== SINGLE STORY CHECK ==========")

story = train_stories[0]

# ---- X ----
processed = preprocess_embeddings(
    stories=[story],
    word_vectors=bert_vectors,
    wordseqs=raw_text,
)
X_story = processed[story]

# ---- Y ----
fmri_dir = f"{DATA_PATH}/{subject}"
Y_story = np.load(f"{fmri_dir}/{story}.npy")

print(f"Story: {story}")
print("X_story shape:", X_story.shape)
print("Y_story shape:", Y_story.shape)

# ---- detect issue ----
if X_story.shape[0] == Y_story.shape[0]:
    print("PERFECT ALIGNMENT")
else:
    print("UNKNOWN MISMATCH")

# ----------------------------

print("\n========== STACKED TRAIN CHECK ==========")

# ---- X stack ----
processed_all = preprocess_embeddings(
    stories=train_stories,
    word_vectors=bert_vectors,
    wordseqs=raw_text,
)

X = np.vstack([processed_all[s] for s in train_stories]).astype(np.float32)

# ---- Y stack ----
Y_list = []
lengths = []

for s in train_stories:
    y = np.load(f"{fmri_dir}/{s}.npy")
    Y_list.append(y)
    lengths.append(y.shape[0])

Y = np.vstack(Y_list)

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nPer-story Y lengths:", lengths[:5], "...")

print("\nExpected rows:", sum(lengths))
print("Actual X rows:", X.shape[0])
print("Actual Y rows:", Y.shape[0])

# ---- final check ----
assert X.shape[0] == Y.shape[0], "X and Y are NOT aligned!"

print("\nFINAL CHECK PASSED: X and Y are aligned.")

========== SINGLE STORY CHECK ==========
Story: mybackseatviewofagreatromance
X_story shape: (384, 3072)
Y_story shape: (384, 94251)
PERFECT ALIGNMENT

========== STACKED TRAIN CHECK ==========
X shape: (384, 3072)
Y shape: (384, 94251)

Per-story Y lengths: [384] ...

Expected rows: 384
Actual X rows: 384
Actual Y rows: 384

FINAL CHECK PASSED: X and Y are aligned.


## Expected outputs

After running all cells successfully, you should see files like:

- `pretrained_bert_embedding_metadata.json`
- `story_level_pretrained_bert/*.npz`
- `s2_train_pretrained_bert_embeddings.npz`
- `s2_test_pretrained_bert_embeddings.npz`
- `s3_train_pretrained_bert_embeddings.npz`
- `s3_test_pretrained_bert_embeddings.npz`